# Re-embed corrected KB rows

54 rows had their **question** text corrected after the collection was built:

| fix | rows | what was wrong |
|---|---|---|
| `iron` -> `آئرن` | 25 | `لوھ` is iron the *metal*, not the dietary mineral |
| `period` -> `ماهواري` | 19 | `عرصو` means "a span of time", not menstruation |
| overlap / nutrition | 10 | localisation rows that also touched the question |

Their **payloads** are already corrected, so the displayed text is right. But
only the *question* is embedded (`embed_and_index.ipynb` encodes
`normalised_questions`), so their **vectors still encode the old wording** --
a woman typing `ماهواري` cannot lexically match a row whose vector says
`عرصو`.

This re-embeds just those 54 and updates their vectors in place. It uses
`update_vectors`, not `upsert`, so payloads are untouched.

Runtime: a few minutes. Needs `QDRANT_URL` and `QDRANT_API_KEY` in Kaggle
Secrets (Add-ons -> Secrets).

In [ ]:
!pip install -q FlagEmbedding qdrant-client pandas 2>/dev/null
print("deps installed")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["QDRANT_URL"] = s.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = s.get_secret("QDRANT_API_KEY")
print("credentials loaded")

In [ ]:
# Pull the corrected knowledge base straight from the branch, so this cannot
# silently run against a stale local copy.
!rm -rf naari-ai
!git clone -q --depth 1 -b sana/phase3 https://github.com/sana200420/naari-ai.git
%cd naari-ai
print("cloned")

In [ ]:
import json, pandas as pd

KB = "knowledge_base/Womens_Health_KB - 2000_final.csv"
kb = pd.read_csv(KB)
ids = json.load(open("data/processed/rows_needing_reembed.json", encoding="utf-8"))["ids"]
rows = kb[kb.id.isin(ids)].copy()
print(f"{len(rows)} rows to re-embed (expected {len(ids)})")
print(rows[["id", "question"]].head(5).to_string(index=False))

In [ ]:
from retrieval.normalize import normalize_sd

# normalize_sd is the ONLY way text reaches a model anywhere in this project;
# embedding a raw string here would produce a vector the live query path can
# never match.
questions = [normalize_sd(q) for q in rows.question.astype(str)]
print(f"normalised {len(questions)} questions")
print(questions[0])

In [ ]:
from FlagEmbedding import BGEM3FlagModel
import torch

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=torch.cuda.is_available())
out = model.encode(questions, return_dense=True, return_sparse=True,
                   return_colbert_vecs=False)
dense = out["dense_vecs"]
sparse = out["lexical_weights"]
print(f"embedded {len(dense)} rows, dense dim {len(dense[0])}")

In [ ]:
from qdrant_client import QdrantClient, models

COLLECTION = "naari_ai_kb"
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

# Sanity check before writing: confirm these ids exist and are the Sindhi rows.
found = client.retrieve(collection_name=COLLECTION, ids=[int(i) for i in rows.id],
                        with_payload=["lang", "question"], with_vectors=False)
langs = {p.payload.get("lang") for p in found}
print(f"found {len(found)}/{len(rows)} points, lang values: {langs}")
assert len(found) == len(rows), "some ids are missing from the collection"
assert langs == {"sd"}, f"expected only Sindhi rows, got {langs}" 

In [ ]:
def to_sparse(w):
    return models.SparseVector(indices=[int(k) for k in w.keys()],
                               values=[float(v) for v in w.values()])

# update_vectors, NOT upsert: upsert would replace the whole point and drop the
# payload we corrected earlier today.
client.update_vectors(
    collection_name=COLLECTION,
    points=[
        models.PointVectors(
            id=int(i),
            vector={"dense": d.tolist(), "sparse": to_sparse(s)},
        )
        for i, d, s in zip(rows.id, dense, sparse)
    ],
    wait=True,
)
print(f"updated vectors for {len(rows)} points")

In [ ]:
# Verify: the payload survived, and a query using the CORRECTED wording now
# retrieves the row it should.
check = client.retrieve(collection_name=COLLECTION, ids=[3, 1048],
                        with_payload=True, with_vectors=False)
for p in check:
    print(p.id, "|", p.payload.get("question"))
    assert p.payload.get("answer"), "payload lost -- this should not happen"

q = normalize_sd("ماهواري عام طور تي ڪيترا ڏينهن هلندي آهي؟")
qv = model.encode([q], return_dense=True, return_sparse=True,
                  return_colbert_vecs=False)
hits = client.query_points(
    collection_name=COLLECTION, query=qv["dense_vecs"][0].tolist(), using="dense",
    query_filter=models.Filter(must=[models.FieldCondition(
        key="lang", match=models.MatchValue(value="sd"))]),
    limit=3, with_payload=["question"]).points
print("\ntop-3 for a ماهواري query:")
for h in hits:
    print(f"  {h.score:.3f}  id={h.id}  {h.payload['question'][:60]}")

## Done

Vectors for the 54 corrected rows now match their text. Nothing else in the
collection was touched, and no payload was rewritten.

Worth re-running afterwards, from the repo root:

```bash
python scripts/tune_tau_high.py        # score distribution shifts after this
python scripts/category_burndown.py    # PCOS/Pregnancy buckets may move
```